In [ ]:
from __future__ import annotations
import os
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Tuple, Sequence, Union
from resolve.utilities import utilities as utils
import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import h5py
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)
from sklearn.utils import shuffle
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import add_self_loops, to_undirected
from torch_geometric.utils import dropout_edge
import matplotlib.pyplot as plt

In [2]:
import hnswlib, numpy as np, torch
from torch_geometric.utils import to_undirected, add_self_loops



In [3]:
def focal_bce_with_logits(logits, targets, alpha=0.5, gamma=2.0):
    p = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
    loss_pos = -alpha * ((1-p)**gamma) * targets * torch.log(p)
    loss_neg = -(1-alpha) * (p**gamma) * (1-targets) * torch.log(1-p)
    return (loss_pos + loss_neg).mean()

In [4]:



# ------------------------------
# Repro
# ------------------------------
from torch import logit


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _get_hdf5_files(path_to_files, config_file):
        return sorted(str(p) for p in path_to_files.glob(f"*.{config_file['simulation_settings']['file_format']}"))

# ------------------------------
# Data utilities + file pipeline
# ------------------------------
def _read_in_from_file(file_path: str, parameter_config: Dict) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Read (theta, phi, y) from HDF5 or CSV and return tensors (x, y).
    - x = [theta | phi] concatenated along last dimension, float32
    - y shaped (N, 1) float32

    parameter_config can provide either 'key'+'selected_indices' (for HDF5) or 'selected_labels' (for CSV):
      'phi':   {'key' or 'selected_labels', 'selected_indices'}
      'theta': {'key' or 'selected_labels', 'selected_indices'}
      'target':{'key' or 'selected_labels', 'selected_indices'}
    """
    if file_path.endswith(('.h5', '.hdf5')):
        with h5py.File(file_path, 'r') as hdf:
            # φ
            phi = hdf[parameter_config['phi']['key']][:, parameter_config['phi']['selected_indices']]
            # θ
            theta = hdf[parameter_config['theta']['key']]
            if len(parameter_config['theta']['selected_indices']) != 0:
                if theta.ndim == 1:
                    theta_vec = theta[parameter_config['theta']['selected_indices']]  # (T,)
                    theta = torch.from_numpy(theta_vec).unsqueeze(0).expand(phi.shape[0], -1)
                else:
                    theta = theta[:, parameter_config['theta']['selected_indices']]
                    theta = torch.from_numpy(theta)
            else:
                theta = torch.from_numpy(theta)

            # y / target
            tgt_ds = hdf[parameter_config['target']['key']]
            if tgt_ds.ndim > 1 and parameter_config['target']['selected_indices'] is not None:
                y = tgt_ds[:, parameter_config['target']['selected_indices']]
            else:
                y = tgt_ds[:].reshape(-1, 1)

        phi = torch.from_numpy(phi)
        y = torch.from_numpy(y)
        x = torch.cat([theta, phi], dim=-1)

    elif file_path.endswith('.csv'):
        # CSV via selected_labels
        df = pd.read_csv(file_path)

        def select_labels(df_: pd.DataFrame, labels: Union[str, Sequence[str]]) -> pd.DataFrame:
            if isinstance(labels, str):
                return df_[[labels]]
            elif isinstance(labels, (list, tuple)):
                return df_[list(labels)]
            else:
                raise ValueError(f"Invalid label type: {type(labels)}")

        phi_df = select_labels(df, parameter_config['phi']['selected_labels'])
        theta_df = select_labels(df, parameter_config['theta']['selected_labels'])
        y_df = select_labels(df, parameter_config['target']['selected_labels'])

        phi = torch.tensor(phi_df.values, dtype=torch.float32)
        theta = torch.tensor(theta_df.values, dtype=torch.float32)
        y = torch.tensor(y_df.values, dtype=torch.float32)

        if y.ndim == 1:
            y = y.unsqueeze(1)

        x = torch.cat([theta, phi], dim=-1)

    else:
        raise ValueError(f"Unsupported file format: {file_path}")

    # ensure float32 tensors
    x = x.contiguous().to(torch.float32)
    y = y.contiguous().to(torch.float32)
    return x, y


def _load_data_to_mem(files: Sequence[str], cfg: Dict) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    X, ys, file_inds = [], [], []
    for i, fp in enumerate(files):
        if not os.path.exists(fp):
            raise FileNotFoundError(fp)
        Xi, yi = _read_in_from_file(fp, cfg)
        X.append(Xi)
        ys.append(yi)
        file_inds.append(torch.full((Xi.size(0),), i, dtype=torch.long))
    x = torch.cat(X, 0).contiguous()
    y = torch.cat(ys, 0).contiguous()
    fidx = torch.cat(file_inds, 0).contiguous()
    return x, y, fidx





    




def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)

In [ ]:
def read_in_data():
        # Example parameter_config (edit to your keys/labels/indices)
        path_to_settings = "./binary-black-hole"
        with open(f"{path_to_settings}/settings.yaml", "r") as f:
            config_file = yaml.safe_load(f)
        sim = config_file["simulation_settings"]
        parameters = {
            "phi":    {"key": "phi",    "label_key": "phi_labels",    "selected_labels": sim["phi_labels"],    "size": len(sim["phi_labels"]),      "selected_indices": None},
            "theta":  {"key": "theta",  "label_key": "theta_headers", "selected_labels": sim["theta_labels"],  "size": len(sim["theta_labels"]),  "selected_indices": None},
            "target": {"key": "target", "label_key": "target_headers","selected_labels": sim["target_labels"], "size": len(sim["target_labels"]), "selected_indices": None},
        }
        files = _get_hdf5_files(Path(config_file["path_settings"]["path_to_files_train"]), config_file)

        if files[0].endswith(('.h5', '.hdf5')):
            parameters["phi"]["selected_indices"] = utils.find_selected_indices(files[0],parameters["phi"])
            parameters["target"]["selected_indices"] = utils.find_selected_indices(files[0],parameters["target"])
            parameters["theta"]["selected_indices"] = utils.find_selected_indices(files[0],parameters["theta"])


        X_t, y_t, _ = _load_data_to_mem(files, parameters)
        # Ensure numpy arrays for kNN builder

        # X is your raw numpy or torch input, shape [N, D]
        #scaler = StandardScaler()

        # Convert to numpy if needed
        X_np = X_t.cpu().numpy() if isinstance(X_t, torch.Tensor) else X_t
        #X_t = X.detach().clone().to(torch.float32) if isinstance(X, torch.Tensor) else torch.tensor(X, dtype=torch.float32)

        # Fit on ALL data (or train split only)
        #X_np = scaler.fit_transform(X_np)

        # Back to torch
        X = X_np
        y_arr = y_t.cpu().numpy().reshape(-1, 1)

        # Ensure binary labels {0,1}. If multi-target or continuous, map/threshold here.
        if y_arr.shape[1] > 1:
            # choose a column or reduce to a binary indicator
            y = (y_arr[:, 0] > 0.5).astype(np.int64)
        else:
            if not np.array_equal(np.unique(y_arr), np.array([0, 1])):
                y = (y_arr[:, 0] > 0.5).astype(np.int64)
            else:
                y = y_arr[:, 0].astype(np.int64)
        
        X, y = shuffle(X, y, random_state=42)
        return X[:100000,:], y[:100000]

In [6]:



def plot_pr_curve(y_true_np, probs_np, outdir="plots", title_prefix="Test"):
    _ensure_dir(outdir)
    precision, recall, _ = precision_recall_curve(y_true_np, probs_np)
    plt.figure()
    plt.plot(recall, precision)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix} PR Curve")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{title_prefix.lower().replace(' ', '_')}_pr_curve.png"), dpi=150)
    plt.close()


def plot_roc_curve(y_true_np, probs_np, outdir="plots", title_prefix="Test"):
    _ensure_dir(outdir)
    fpr, tpr, _ = roc_curve(y_true_np, probs_np)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"{title_prefix} ROC Curve")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{title_prefix.lower().replace(' ', '_')}_roc_curve.png"), dpi=150)
    plt.close()


def plot_score_hist(probs_np, y_true_np, outdir="plots", title_prefix="Test"):
    _ensure_dir(outdir)
    plt.figure()
    plt.hist(probs_np[y_true_np == 0], bins=50, alpha=0.7, label="neg")
    plt.hist(probs_np[y_true_np == 1], bins=50, alpha=0.7, label="pos")
    plt.xlabel("Predicted probability")
    plt.ylabel("Count")
    plt.yscale('log')
    plt.title(f"{title_prefix} Score Histogram")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{title_prefix.lower().replace(' ', '_')}_score_hist.png"), dpi=150)
    plt.close()







    
        


In [7]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
"""
class SupConEncoder(nn.Module):
    def __init__(self, in_dim=5, hid=128, emb_dim=64, p_drop=0.1):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_dim, hid), nn.LayerNorm(hid), nn.ReLU(), nn.Dropout(p_drop),
            nn.Linear(hid, hid),    nn.LayerNorm(hid), nn.ReLU(), nn.Dropout(p_drop),
        )
        self.head = nn.Sequential(
            nn.Linear(hid, hid), nn.ReLU(),
            nn.Linear(hid, emb_dim)
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.head(h)
        z = F.normalize(z, dim=-1)  # L2-normalized embeddings
        return z

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, z, y):
        # z: [B,D] (normalized), y: [B] int labels {0,1}
        B = z.size(0)
        sim = (z @ z.t()) / self.tau                      # [B,B]
        mask = torch.ones_like(sim, dtype=torch.bool)
        mask.fill_diagonal_(False)                        # no self-contrast
        y = y.view(-1,1)
        pos_mask = (y == y.t()) & mask                    # positives by label
        # log-softmax over all non-self entries
        sim = sim.masked_fill(~mask, -1e9)
        log_prob = sim - torch.logsumexp(sim, dim=1, keepdim=True)  # [B,B]
        # average over positives per anchor
        pos_count = pos_mask.sum(1).clamp_min(1)
        loss = -(log_prob.masked_select(pos_mask).sum() / pos_count.sum())
        return loss
"""

'\nclass SupConEncoder(nn.Module):\n    def __init__(self, in_dim=5, hid=128, emb_dim=64, p_drop=0.1):\n        super().__init__()\n        self.backbone = nn.Sequential(\n            nn.Linear(in_dim, hid), nn.LayerNorm(hid), nn.ReLU(), nn.Dropout(p_drop),\n            nn.Linear(hid, hid),    nn.LayerNorm(hid), nn.ReLU(), nn.Dropout(p_drop),\n        )\n        self.head = nn.Sequential(\n            nn.Linear(hid, hid), nn.ReLU(),\n            nn.Linear(hid, emb_dim)\n        )\n\n    def forward(self, x):\n        h = self.backbone(x)\n        z = self.head(h)\n        z = F.normalize(z, dim=-1)  # L2-normalized embeddings\n        return z\n\nclass SupConLoss(nn.Module):\n    def __init__(self, temperature=0.07):\n        super().__init__()\n        self.tau = temperature\n\n    def forward(self, z, y):\n        # z: [B,D] (normalized), y: [B] int labels {0,1}\n        B = z.size(0)\n        sim = (z @ z.t()) / self.tau                      # [B,B]\n        mask = torch.ones_like(s

In [8]:
import torch, torch.nn as nn
import torch.nn.functional as F

class SupConEncoder(nn.Module):
    def __init__(self, in_dim, enc_dim=256, proj_dim=128):
        super().__init__()
        # backbone for tabular (use yours if you already have one)
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 512), nn.ReLU(),
            nn.Linear(512, enc_dim), nn.ReLU(),
        )
        # projection head used ONLY for SupCon
        self.proj = nn.Sequential(
            nn.Linear(enc_dim, enc_dim), nn.ReLU(),
            nn.Linear(enc_dim, proj_dim),
        )

    def forward(self, x, *, return_proj=True):
        h = self.encoder(x)                 # [B, enc_dim]
        if not return_proj:
            return h
        z = self.proj(h)                    # [B, proj_dim]
        z = F.normalize(z, dim=-1)          # cosine space
        return z

In [9]:
import torch
import torch.nn.functional as F

class SupConLoss(torch.nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, z, y):
        """
        z: [B, D] normalized embeddings
        y: [B] int labels (0/1)
        """
        z = F.normalize(z, dim=-1)                           # cosine
        logits = (z @ z.t()) / self.tau                      # [B,B]
        # mask out self-comparisons
        logits = logits - torch.eye(z.size(0), device=z.device) * 1e9

        # positive mask (same class & class==1 to avoid pulling negatives together)
        y = y.view(-1, 1)
        pos_mask = (y == y.t()) & (y == 1)

        # for each anchor, logsumexp over all others (denom)
        log_denom = torch.logsumexp(logits, dim=1, keepdim=True)  # [B,1]

        # numerator: only positives
        # add -inf to non-positives so exp() zeroes them out
        num = torch.logsumexp(torch.where(pos_mask, logits, torch.full_like(logits, -1e9)), dim=1)

        # avoid anchors with no positives in batch
        valid = pos_mask.any(dim=1)
        loss = -(num[valid] - log_denom[valid].squeeze(1)).mean()
        return loss

In [10]:
def make_label_aware_batches(y_np, batch_size=2048, min_pos=128, seed=0):
    rng = np.random.default_rng(seed)
    idx_pos = np.where(y_np==1)[0]
    idx_neg = np.where(y_np==0)[0]
    while True:
        # sample positives with replacement if necessary
        p = rng.choice(idx_pos, size=min_pos, replace=(len(idx_pos) < min_pos))
        n = rng.choice(idx_neg, size=batch_size - min_pos, replace=False)
        batch_idx = np.concatenate([p, n])
        rng.shuffle(batch_idx)
        yield batch_idx

In [11]:
@torch.no_grad()
def pick_hard_negs_in_batch(z, y, max_negs_per_pos=4):
    # z: [B,D], y:[B], returns a boolean mask of which negatives to keep
    sim = z @ z.t()
    B = z.size(0)
    keep = torch.zeros(B, B, dtype=torch.bool, device=z.device)
    for i in range(B):
        pos = (y == y[i])
        neg = ~pos
        neg[i] = False
        # pick top hard negatives by similarity
        sim_i = sim[i].clone()
        sim_i[~neg] = -1e9
        idx = torch.topk(sim_i, k=min(max_negs_per_pos, neg.sum().item()))[1]
        keep[i, idx] = True
        # keep all positives for anchor i
        keep[i, pos & (torch.arange(B, device=z.device)!=i)] = True
    return keep  # use to mask logsumexp and pos sets if you want tighter mining

In [12]:
from sklearn.metrics import average_precision_score, roc_auc_score

def train_supcon(X_np, y_np, epochs=10, batch_size=2048, min_pos=128, lr=3e-4, wd=1e-4, tau=0.02, device="cuda"):
    model = SupConEncoder(in_dim=X_np.shape[1]).to(device)
    crit = SupConLoss(temperature=tau)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    sampler = make_label_aware_batches(y_np, batch_size=batch_size, min_pos=min_pos, seed=0)
    steps_per_epoch = max(1, (len(X_np)//batch_size))

    model.train()
    for ep in range(1, epochs+1):
        running = 0.0
        for _ in range(steps_per_epoch):
            bidx = next(sampler)
            xb = torch.from_numpy(X_np[bidx]).float().to(device)
            yb = torch.from_numpy(y_np[bidx]).long().to(device)
            z = model(xb)
            loss = crit(z, yb)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running += loss.item()
        print(f"Epoch {ep:03d} | SupCon loss {running/steps_per_epoch:.4f}")
    model.eval()
    return model

In [13]:
from sklearn.neighbors import NearestNeighbors

@torch.no_grad()
def compute_embeddings(model, X_np, device="cuda", batch=65536):
    Z = []
    for i in range(0, len(X_np), batch):
        xb = torch.from_numpy(X_np[i:i+batch]).float().to(device)
        zb = model(xb).cpu().numpy()
        Z.append(zb)
    Z = np.vstack(Z)
    # already L2-normalized by model; normalize again for safety
    Z = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12)
    return Z

def knn_purity(Z, y_np, K=10, metric='cosine'):
    nn = NearestNeighbors(n_neighbors=K+1, metric=metric).fit(Z)
    d, idx = nn.kneighbors(Z, n_neighbors=K+1)   # includes self at [:,0]
    idx = idx[:,1:]                               # drop self
    nbr_labels = y_np[idx]                        # [N,K]
    # purity for positives only
    pos = (y_np==1)
    if pos.sum()==0: return 0.0, 0.0
    pos_purity = nbr_labels[pos].mean()           # fraction of positives among neighbors
    # probability of at least one positive neighbor
    p_at_least_one = 1.0 - (1.0 - nbr_labels[pos].mean(axis=1)).prod(axis=0)  # not exact; alternative below:
    # better: compute per-row: (nbr_labels[pos].sum(axis=1) > 0).mean()
    p_at_least_one = (nbr_labels[pos].sum(axis=1) > 0).mean()
    return float(pos_purity), float(p_at_least_one)

In [14]:
device = "cpu"  # or "cpu"
X_all, y_all = read_in_data()  # returns raw arrays (no scaling)
X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)
X_tr, X_va, y_tr, y_va = train_test_split(X_tr, y_tr, test_size=0.2, stratify=y_tr, random_state=42)

scaler = StandardScaler().fit(X_tr)
X_tr = scaler.transform(X_tr)
X_va = scaler.transform(X_va)
X_te = scaler.transform(X_te)

model = train_supcon(
    X_tr, y_tr,
    epochs=10,
    batch_size=1024,   # lower on CPU (e.g., 512)
    min_pos=128,       # ensure your batch has enough positives
    lr=3e-4, wd=1e-4, tau=0.7,
    device=device
)

Epoch 001 | SupCon loss 0.9844
Epoch 002 | SupCon loss 0.7710
Epoch 003 | SupCon loss 0.7174
Epoch 004 | SupCon loss 0.6753
Epoch 005 | SupCon loss 0.6683
Epoch 006 | SupCon loss 0.6552
Epoch 007 | SupCon loss 0.6571
Epoch 008 | SupCon loss 0.6857
Epoch 009 | SupCon loss 0.6392
Epoch 010 | SupCon loss 0.6022


In [15]:
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score

def knn_classifier_pr(Z, y_np, K=32, metric='cosine'):
    nn = NearestNeighbors(n_neighbors=K, metric=metric).fit(Z)
    distances, idx = nn.kneighbors(Z)
    preds = y_np[idx].mean(axis=1)  # proportion of positives among neighbors
    ap = average_precision_score(y_np, preds)
    roc = roc_auc_score(y_np, preds)
    return ap, roc

In [16]:
Z_tr = compute_embeddings(model, X_tr, device=device)
Z_va = compute_embeddings(model, X_va, device=device)
Z_te = compute_embeddings(model, X_te, device=device)

from sklearn.neighbors import NearestNeighbors
import numpy as np

def knn_metrics(Z_ref, y_ref, Z_query, y_query, K=32, metric='cosine', drop_self=False):
    nn = NearestNeighbors(n_neighbors=K + (1 if drop_self else 0), metric=metric).fit(Z_ref)
    dist, idx = nn.kneighbors(Z_query)
    if drop_self:
        # only matters if Z_query is Z_ref; remove the first neighbor (self)
        same = (Z_query is Z_ref) or (Z_query.shape == Z_ref.shape and np.allclose(Z_query, Z_ref))
        if same:
            idx = idx[:, 1:]
        else:
            idx = idx[:, :K]
    # kNN scores = mean positive rate among neighbors
    knn_score = y_ref[idx].mean(axis=1)
    ap = average_precision_score(y_query, knn_score)
    roc = roc_auc_score(y_query, knn_score)
    return ap, roc

# Evaluate (no leakage):
ap_va, roc_va = knn_metrics(Z_tr, y_tr, Z_va, y_va, K=32, metric='cosine')
ap_te, roc_te = knn_metrics(Z_tr, y_tr, Z_te, y_te, K=32, metric='cosine')
print(f"VAL  kNN(PR-AUC)={ap_va:.4f} | kNN(ROC-AUC)={roc_va:.4f}")
print(f"TEST kNN(PR-AUC)={ap_te:.4f} | kNN(ROC-AUC)={roc_te:.4f}")

VAL  kNN(PR-AUC)=0.1300 | kNN(ROC-AUC)=0.8736
TEST kNN(PR-AUC)=0.1641 | kNN(ROC-AUC)=0.8614


In [17]:
def purity_at_k(Z_ref, y_ref, Z_query, y_query, K=10, metric='cosine', drop_self=False):
    nn = NearestNeighbors(n_neighbors=K + (1 if drop_self else 0), metric=metric).fit(Z_ref)
    _, idx = nn.kneighbors(Z_query)
    if drop_self:
        # only when querying the same set
        same = (Z_query is Z_ref) or (Z_query.shape == Z_ref.shape and np.allclose(Z_query, Z_ref))
        idx = idx[:, 1:] if same else idx[:, :K]
    nbr_labels = y_ref[idx]                           # [Nq, K]
    pos = (y_query == 1)
    if pos.sum() == 0: return 0.0, 0.0
    pos_purity = nbr_labels[pos].mean()               # mean fraction positives among K
    p_ge1 = (nbr_labels[pos].sum(axis=1) > 0).mean()  # at least one positive
    return float(pos_purity), float(p_ge1)

p10_va, hit_va = purity_at_k(Z_tr, y_tr, Z_va, y_va, K=10, metric='cosine')
p10_te, hit_te = purity_at_k(Z_tr, y_tr, Z_te, y_te, K=10, metric='cosine')
print(f"VAL  pos-purity@10={p10_va:.3f} | P(≥1 pos@10)={hit_va:.3f}")
print(f"TEST pos-purity@10={p10_te:.3f} | P(≥1 pos@10)={hit_te:.3f}")

VAL  pos-purity@10=0.157 | P(≥1 pos@10)=0.684
TEST pos-purity@10=0.172 | P(≥1 pos@10)=0.664


In [18]:

# get H (pre-projection) embeddings
@torch.no_grad()
def encode_h(model, X_np, device="cpu", batch=65536):
    H = []
    for i in range(0, len(X_np), batch):
        xb = torch.from_numpy(X_np[i:i+batch]).float().to(device)
        hb = model(xb, return_proj=False).cpu().numpy()
        H.append(hb)
    return np.vstack(H)

H_tr, H_va, H_te = encode_h(model, X_tr), encode_h(model, X_va), encode_h(model, X_te)

# train probe
import torch.nn.functional as F, torch.nn as nn, torch
def train_probe(H, y, lr=1e-3, wd=1e-4, epochs=10, device="cpu"):
    Ht = torch.from_numpy(H).float().to(device)
    yt = torch.from_numpy(y).float().to(device)
    head = nn.Linear(H.shape[1], 1).to(device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        opt.zero_grad()
        logits = head(Ht).squeeze(1)
        loss = F.binary_cross_entropy_with_logits(logits, yt)
        loss.backward(); opt.step()
    head.eval(); return head

head = train_probe(H_tr, y_tr)

In [19]:
@torch.no_grad()
def probe_logits(head, H, device="cuda"):
    return head(torch.from_numpy(H).float().to(device)).squeeze(1)

logits_va = probe_logits(head, H_va, device=device)
logits_te = probe_logits(head, H_te, device=device)

probs_va = torch.sigmoid(logits_va).cpu().numpy()
probs_te = torch.sigmoid(logits_te).cpu().numpy()

In [20]:
from sklearn.metrics import average_precision_score, roc_auc_score

print("VAL  PR-AUC=%.4f | ROC-AUC=%.4f" % (
    average_precision_score(y_va, probs_va),
    roc_auc_score(y_va, probs_va)
))
print("TEST PR-AUC=%.4f | ROC-AUC=%.4f" % (
    average_precision_score(y_te, probs_te),
    roc_auc_score(y_te, probs_te)
))

VAL  PR-AUC=0.0257 | ROC-AUC=0.8373
TEST PR-AUC=0.0244 | ROC-AUC=0.8390


In [21]:
# 1) Get z per split (use your existing compute_embeddings but per split)
Z_tr = compute_embeddings(model, X_tr, device=device)  # [Ntr, d], L2-normalized
Z_va = compute_embeddings(model, X_va, device=device)
Z_te = compute_embeddings(model, X_te, device=device)

# 2) Train a weighted logistic head on z (handles 0.73% positives)
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np

def train_z_probe(Z, y, lr=1e-3, wd=1e-4, epochs=10, device="cuda"):
    Zt = torch.from_numpy(Z).float().to(device)
    yt = torch.from_numpy(y).float().to(device)

    head = nn.Linear(Z.shape[1], 1, bias=True).to(device)

    # class imbalance weight: pos_weight = N_neg / N_pos
    n_pos = float((y==1).sum())
    n_neg = float((y==0).sum())
    pos_weight = torch.tensor([n_neg / max(1.0, n_pos)], device=device)

    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        opt.zero_grad()
        logits = head(Zt).squeeze(1)
        loss = F.binary_cross_entropy_with_logits(logits, yt, pos_weight=pos_weight)
        loss.backward()
        opt.step()
    head.eval()
    return head

head_z = train_z_probe(Z_tr, y_tr, epochs=10, lr=1e-3, wd=1e-4, device=device)

@torch.no_grad()
def infer(head, Z, device="cuda"):
    return head(torch.from_numpy(Z).float().to(device)).squeeze(1)

logits_va = infer(head_z, Z_va, device=device)
logits_te = infer(head_z, Z_te, device=device)

from sklearn.metrics import average_precision_score, roc_auc_score
probs_va = torch.sigmoid(logits_va).cpu().numpy()
probs_te = torch.sigmoid(logits_te).cpu().numpy()

print("VAL  PR-AUC=%.4f | ROC-AUC=%.4f" % (
    average_precision_score(y_va, probs_va),
    roc_auc_score(y_va, probs_va)
))
print("TEST PR-AUC=%.4f | ROC-AUC=%.4f" % (
    average_precision_score(y_te, probs_te),
    roc_auc_score(y_te, probs_te)
))

VAL  PR-AUC=0.0933 | ROC-AUC=0.9186
TEST PR-AUC=0.0928 | ROC-AUC=0.8954


In [22]:
class MLPProbe(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )
    def forward(self, Z): return self.net(Z).squeeze(1)

def train_mlp_probe(Z, y, epochs=10, lr=1e-3, wd=1e-4, device="cuda"):
    Zt = torch.from_numpy(Z).float().to(device)
    yt = torch.from_numpy(y).float().to(device)
    head = MLPProbe(Z.shape[1]).to(device)
    # handle imbalance
    pos_weight = torch.tensor([(y==0).sum()/max(1,(y==1).sum())], device=device)
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=wd)
    for _ in range(epochs):
        opt.zero_grad()
        logits = head(Zt)
        loss = F.binary_cross_entropy_with_logits(logits, yt, pos_weight=pos_weight)
        loss.backward(); opt.step()
    head.eval(); return head

In [23]:
head_z_mlp = train_mlp_probe(Z_tr, y_tr, epochs=10, lr=1e-3, wd=1e-4, device=device)

@torch.no_grad()
@torch.no_grad()
def infer(head, Z, device="cuda"):
    X = torch.from_numpy(Z).float().to(device)
    out = head(X)
    return out.reshape(-1)   # works for [N] or [N,1]

logits_va = infer(head_z_mlp, Z_va, device=device)
logits_te = infer(head_z_mlp, Z_te, device=device)

from sklearn.metrics import average_precision_score, roc_auc_score
probs_va = torch.sigmoid(logits_va).cpu().numpy()
probs_te = torch.sigmoid(logits_te).cpu().numpy()

print("VAL  PR-AUC=%.4f | ROC-AUC=%.4f" % (
    average_precision_score(y_va, probs_va),
    roc_auc_score(y_va, probs_va)
))
print("TEST PR-AUC=%.4f | ROC-AUC=%.4f" % (
    average_precision_score(y_te, probs_te),
    roc_auc_score(y_te, probs_te)
))

VAL  PR-AUC=0.1509 | ROC-AUC=0.9466
TEST PR-AUC=0.1486 | ROC-AUC=0.9396


In [25]:
class TempScaler(nn.Module):
    def __init__(self): super().__init__(); self.log_t = nn.Parameter(torch.zeros(()))
    def forward(self, x): return x / self.log_t.exp()

def fit_temp(logits_val, y_val):
    ts = TempScaler().to(logits_val.device)
    opt = torch.optim.LBFGS(ts.parameters(), lr=0.1, max_iter=200)
    y = y_val.float()
    def closure():
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(ts(logits_val), y)
        loss.backward(); return loss
    opt.step(closure); return ts

# logits from your MLP probe:
logits_va = infer(head_z_mlp, Z_va, device=device)
logits_te = infer(head_z_mlp, Z_te, device=device)
ts = fit_temp(logits_va, torch.as_tensor(y_va, device=logits_va.device))
probs_te_cal = torch.sigmoid(ts(logits_te)).detach().cpu().numpy()

print("TEST (cal) PR-AUC=%.4f | ROC-AUC=%.4f" % (
    average_precision_score(y_te, probs_te_cal),
    roc_auc_score(y_te, probs_te_cal)
))

TEST (cal) PR-AUC=0.1486 | ROC-AUC=0.9396


In [29]:
from sklearn.neighbors import NearestNeighbors
import numpy as np, torch, torch.nn.functional as F

# Assume Z_tr, y_tr, Z_va, y_va, Z_te, y_te (L2-normalized z); head_mlp trained on Z_tr
@torch.no_grad()
def logits(head, Z, device="cpu"):
    return head(torch.from_numpy(Z).float().to(device)).squeeze(1)
@torch.no_grad()
def logits(head, Z, device="cpu"):
    X = torch.from_numpy(Z).float().to(device)
    out = head(X)                  # could be [N] or [N,1]
    return out.reshape(-1)         # always -> [N]
# kNN positives proportion using train as reference
def knn_posprop(Z_ref, y_ref, Z_query, K=32):
    nn = NearestNeighbors(n_neighbors=K, metric='cosine').fit(Z_ref)
    _, idx = nn.kneighbors(Z_query)
    return y_ref[idx].mean(axis=1)

log_va = logits(head_z_mlp, Z_va, device=device); pr_va = torch.sigmoid(log_va).detach().cpu().numpy()
log_te = logits(head_z_mlp, Z_te, device=device); pr_te = torch.sigmoid(log_te).detach().cpu().numpy()
knn_va = knn_posprop(Z_tr, y_tr, Z_va, K=32)
knn_te = knn_posprop(Z_tr, y_tr, Z_te, K=32)

def evaluate_blend(alpha):
    s_va = alpha*pr_va + (1-alpha)*knn_va
    s_te = alpha*pr_te + (1-alpha)*knn_te
    from sklearn.metrics import average_precision_score, roc_auc_score
    return (
        average_precision_score(y_va, s_va), roc_auc_score(y_va, s_va),
        average_precision_score(y_te, s_te), roc_auc_score(y_te, s_te),
    )

for a in [0.2, 0.3, 0.4, 0.5, 0.7]:
    apv, rocv, apt, roct = evaluate_blend(a)
    print(f"alpha={a:.1f} | VAL AP={apv:.4f} ROC={rocv:.4f} | TEST AP={apt:.4f} ROC={roct:.4f}")

alpha=0.2 | VAL AP=0.1566 ROC=0.9467 | TEST AP=0.1930 ROC=0.9397
alpha=0.3 | VAL AP=0.1566 ROC=0.9467 | TEST AP=0.1930 ROC=0.9397
alpha=0.4 | VAL AP=0.1566 ROC=0.9467 | TEST AP=0.1930 ROC=0.9397
alpha=0.5 | VAL AP=0.1566 ROC=0.9467 | TEST AP=0.1930 ROC=0.9397
alpha=0.7 | VAL AP=0.1566 ROC=0.9467 | TEST AP=0.1931 ROC=0.9397


In [30]:
from scipy.stats import spearmanr
rho_va = spearmanr(pr_va, knn_va).correlation
rho_te = spearmanr(pr_te, knn_te).correlation
print(rho_va, rho_te)  # if ~0.98–1.00 → redundant

0.3887483299303361 0.3878384722921036


In [31]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

def knn_density(Z_ref, Z_query, k=32):
    nn = NearestNeighbors(n_neighbors=k, metric='cosine').fit(Z_ref)
    _, idx = nn.kneighbors(Z_query)  # [N_query, k]
    sims = np.sum(Z_query[:, None, :] * Z_ref[idx], axis=2)  # cosines
    return sims.mean(axis=1)

In [32]:
dens_va = knn_density(Z_tr, Z_va)
dens_te = knn_density(Z_tr, Z_te)

In [33]:
for a in [0.1, 0.2, 0.3, 0.4, 0.5, 0.7]:
    s_va = a*pr_va + (1-a)*dens_va
    s_te = a*pr_te + (1-a)*dens_te

    print(a,
          average_precision_score(y_va, s_va),
          roc_auc_score(y_va, s_va),
          average_precision_score(y_te, s_te),
          roc_auc_score(y_te, s_te))

0.1 0.13758240289889287 0.707401127102723 0.14491105799909204 0.6770282069962399
0.2 0.14348003187720149 0.7468165418683219 0.1493878718351508 0.7100085313622758
0.3 0.14581086036976237 0.7597540906426927 0.1509825369061336 0.7224306339949735
0.4 0.14806026116794577 0.7701169794562724 0.1521701093686345 0.7349806068568592
0.5 0.1491201015167964 0.7699467844105699 0.15228896056929492 0.7350583746112375
0.7 0.15391038077627328 0.777512696836566 0.15543784167735875 0.7620873963172852


In [34]:
def topk_metrics(score, y, K=10):
    idx = np.argsort(-score)
    topk = y[idx][:K]
    purity = topk.mean()
    atleast1 = (topk.sum() > 0).astype(float)
    return purity, atleast1

In [35]:
s_va = 0.7*pr_va + 0.3*dens_va
s_te = 0.7*pr_te + 0.3*dens_te

print("VAL", topk_metrics(s_va, y_va))
print("TEST", topk_metrics(s_te, y_te))

VAL (0.2, 1.0)
TEST (0.2, 1.0)


In [36]:
for alpha in [0.4, 0.5, 0.6]:
    s_va = alpha*pr_va + (1-alpha)*dens_va
    s_te = alpha*pr_te + (1-alpha)*dens_te
    print(alpha, topk_metrics(s_va, y_va), topk_metrics(s_te, y_te))

0.4 (0.2, 1.0) (0.2, 1.0)
0.5 (0.2, 1.0) (0.0, 0.0)
0.6 (0.2, 1.0) (0.1, 1.0)


In [39]:
def z(x):
    return (x - x.mean()) / (x.std() + 1e-8)

pr_va_z = z(pr_va)
pr_te_z = z(pr_te)
dens_va_z = z(dens_va)
dens_te_z = z(dens_te)
for alpha in [0.2, 0.3, 0.4, 0.5, 0.6]:
    s_va = alpha*pr_va_z + (1-alpha)*dens_va_z
    s_te = alpha*pr_te_z + (1-alpha)*dens_te_z
    print(alpha, topk_metrics(s_va,y_va), topk_metrics(s_te,y_te))

0.2 (0.2, 1.0) (0.3, 1.0)
0.3 (0.2, 1.0) (0.2, 1.0)
0.4 (0.2, 1.0) (0.1, 1.0)
0.5 (0.2, 1.0) (0.0, 0.0)
0.6 (0.2, 1.0) (0.2, 1.0)


In [40]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
import torch, torch.nn.functional as F

def zscore(a): 
    a = np.asarray(a); return (a - a.mean()) / (a.std() + 1e-8)

@torch.no_grad()
def logits(head, Z, device="cpu"):
    out = head(torch.from_numpy(Z).float().to(device))
    return out.reshape(-1).cpu().numpy()

def knn_density(Z_ref, Z_query, k=30):
    nn = NearestNeighbors(n_neighbors=k, metric='cosine').fit(Z_ref)
    _, idx = nn.kneighbors(Z_query)
    sims = np.sum(Z_query[:,None,:] * Z_ref[idx], axis=2)
    return sims.mean(axis=1)

def blended_score(Z_ref, y_ref, Z_query, head, alpha=0.2, k=30, device="cpu"):
    pr = torch.sigmoid(torch.tensor(logits(head, Z_query, device=device))).numpy()
    dens = knn_density(Z_ref, Z_query, k=k)
    pr_z, dens_z = zscore(pr), zscore(dens)
    return alpha*pr_z + (1-alpha)*dens_z   # final score

In [42]:
from sklearn.metrics import average_precision_score, roc_auc_score
s_te = blended_score(Z_tr, y_tr, Z_te, head_z_mlp, alpha=0.2, k=30, device=device)
print("TEST AP/ROC:", average_precision_score(y_te, s_te), roc_auc_score(y_te, s_te))

TEST AP/ROC: 0.15008595384504345 0.7164935342974095


In [43]:
def topk_metrics(score, y, K=10):
    idx = np.argsort(-score)
    topk = y[idx][:K]
    return float(topk.mean()), float((topk.sum()>0))

print("TEST purity@10, P>=1@10:", topk_metrics(s_te, y_te, K=10))

TEST purity@10, P>=1@10: (0.3, 1.0)


In [ ]:
def rerank_topM(Z_ref, y_ref, Z_query, base_scores_ref, alpha=0.3, M=100, k=30):
    nnM = NearestNeighbors(n_neighbors=M, metric='cosine').fit(Z_ref)
    _, idxM = nnM.kneighbors(Z_query)                 # [Nq,M]
    # density within candidate set
    dens = []
    for row in idxM:
        Zc = Z_ref[row]
        S = Zc @ Zc.T
        np.fill_diagonal(S, -1e9)
        dens.append(np.mean(np.sort(S, axis=1)[:, -k:], axis=1))
    dens = np.vstack(dens)                            # [Nq,M]
    scoresM = base_scores_ref[idxM]                   # [Nq,M]
    s_blend = alpha*scoresM + (1-alpha)*dens
    order = np.argsort(-s_blend, axis=1)[:, :10]      # take K=10
    topK = np.take_along_axis(idxM, order, axis=1)
    nbr_labels = y_ref[topK]
    pos = (y_query==1)
    return float(nbr_labels[pos].mean()), float((nbr_labels[pos].sum(axis=1)>0).mean())